# Full Pipeline — RQ-VAE + T5 End-to-End (Colab)

**何时用这个 notebook**：

- 你改了 embedding（换模型 / 换 prompt / 换字段），需要从 RQ-VAE 开始**整条 pipeline 重跑**
- 一次 upload → 一次跑完 → 一次下载结果，不用在两个 notebook 之间来回切

**需要上传**：
- `embedding/item_embeddings_raw.npy`（本地 ~74 MB，nomic 768d float64）

**产物**：
- `checkpoints/rqvae_best.pt`
- `embedding/semantic_ids_rqvae.npy`
- `checkpoints/best_model_t5.pt`
- 最终 test Recall@K / NDCG@K 打印在 Cell 8 输出里

**预计耗时（A100）**：upload 1 min + RQ-VAE 1-2 h + T5 训练 2-3 h + eval 5 min ≈ **4-6 小时**

⚠️ 长 session 风险：Colab 偶尔掉线。Cell 7 前会把 RQ-VAE 的中间产物打包下载一次做安全网，即使 T5 阶段中断，RQ-VAE 不用重跑。

---

## 运行顺序
1. **环境**（Cell 1–3）：GPU → deps → clone
2. **上传 embedding**（Cell 4）：单次 74 MB
3. **RQ-VAE 阶段**（Cell 5–6）：train + generate SIDs
4. **中间安全网**（Cell 7）：下载 RQ-VAE 产物
5. **T5 阶段**（Cell 8–9）：train + evaluate
6. **最终下载**（Cell 10，可选）

In [ ]:
# Cell 1：GPU 检查
import torch
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU 型号: {name}')
    print(f'显存:     {mem:.1f} GB')
    if 'A100' in name:
        print('\n✅ A100，保持默认 batch_size=512')
    elif mem < 20:
        print('\n⚠️  显存偏小。RQ-VAE batch=1024 和 T5 batch=512 可能 OOM。')
        print('    OOM 时分别改 embedding/rqvae.py 的 BATCH_SIZE 和 train.py 的 CONFIG["batch_size"]。')
else:
    print('\n❌ 没 GPU。Runtime → Change runtime type → GPU')

In [ ]:
# Cell 2：安装依赖
!pip install transformers scikit-learn -q
print('依赖安装完成 ✅')

In [ ]:
# Cell 3：clone 仓库（或 pull 更新）
import os
REPO = 'Generative-Sequential-Recommendation'
if not os.path.exists(REPO):
    !git clone https://github.com/rhine-yangrui/Generative-Sequential-Recommendation.git
else:
    print('repo 已存在，执行 git pull 同步最新提交')
    !cd {REPO} && git pull
%cd {REPO}
!git log --oneline -3

In [ ]:
# Cell 4：上传 item_embeddings_raw.npy（~74 MB）
# 点击 "Choose Files" 按钮，选本地项目 embedding/ 目录下的 item_embeddings_raw.npy
import os
from google.colab import files

os.makedirs('embedding', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

print('请上传 item_embeddings_raw.npy（本地项目 embedding/ 目录里）')
uploaded = files.upload()
for fname in uploaded:
    if fname != 'item_embeddings_raw.npy':
        print(f'⚠️  文件名不对：收到 {fname!r}，期望 item_embeddings_raw.npy')
        continue
    dest = f'embedding/{fname}'
    os.rename(fname, dest)
    print(f'✅ 已移动到 {dest}  ({os.path.getsize(dest)/1e6:.1f} MB)')

# Sanity check
import numpy as np
raw = np.load('embedding/item_embeddings_raw.npy', allow_pickle=True).item()
vals = list(raw.values())
print(f'\n✅ 加载成功：{len(raw)} items × {vals[0].shape[0]}d  (dtype: {vals[0].dtype})')
norms = [float(np.linalg.norm(v)) for v in vals[:500]]
print(f'   norm mean={np.mean(norms):.3f}  std={np.std(norms):.3f}')
assert len(raw) == 12101, f'期望 12101 items，实际 {len(raw)}'
assert vals[0].shape == (768,), f'期望 768d，实际 {vals[0].shape}'

---
## RQ-VAE 阶段（~1-2 h on A100）

配置见 `embedding/rqvae.py`：K_LEVELS=[256,256,256], batch=1024, lr=1e-3, 3000 epoch,
最后一层 Sinkhorn ε=0.003。保存最高 unique_rate 的 checkpoint 到 `checkpoints/rqvae_best.pt`。

盯几个信号：
- `loss_recon` 稳定下降
- `unique_rate` 逐步上升到 ~95%
- codebook usage c0/c1/c2 都应该 >90%（如果有一层掉到 50% 说明崩了）

In [ ]:
# Cell 5：训练 RQ-VAE
!python embedding/rqvae.py

In [ ]:
# Cell 6：argmin 量化 + c4 collision resolution
!python embedding/generate_rqvae_ids.py

# Sanity check 产物
import numpy as np
sids = np.load('embedding/semantic_ids_rqvae.npy', allow_pickle=True).item()
vals = list(sids.values())
assert len(vals[0]) == 4
unique = len({tuple(v) for v in vals})
print(f'\n✅ {len(sids)} items，唯一 4-tuple: {unique}')
for i in range(4):
    used = len({v[i] for v in vals})
    print(f'   c{i} usage: {used}')

---
## 中间安全网：下载 RQ-VAE 产物

T5 训练是下一个长 session（2-3 h）。在进入之前把 RQ-VAE 的两个产物下载到本地——万一 T5 阶段 Colab 掉线，下次可以从上传 SIDs 直接进 T5 训练，不用重跑 RQ-VAE。

In [ ]:
# Cell 7：下载 RQ-VAE 产物作为安全网
import os
from google.colab import files

for fpath in ['checkpoints/rqvae_best.pt', 'embedding/semantic_ids_rqvae.npy']:
    if os.path.exists(fpath):
        files.download(fpath)
        print(f'✅ {fpath}  ({os.path.getsize(fpath)/1e6:.1f} MB)')
    else:
        print(f'❌ 不存在: {fpath}')

---
## T5 阶段（~2-3 h on A100）

配置见 `train.py` CONFIG：batch=512, lr=1e-4, 200 epoch, patience=10, val_every=2。
早停通常在 ~130 epoch 附近触发。保存最高 val NDCG@10 的 ckpt 到 `checkpoints/best_model_t5.pt`。

In [ ]:
# Cell 8：训练 T5（默认用 semantic_ids_rqvae.npy，对应主线不带参数）
!python train.py

In [ ]:
# Cell 9：all-rank 评估
!python evaluate.py

---
## 最终下载（可选）

把 T5 checkpoint 也拖回本地。`rqvae_best.pt` 和 `semantic_ids_rqvae.npy` 在 Cell 7 已经下过了，这里可以跳过它们。

In [ ]:
# Cell 10：下载 T5 checkpoint
import os
from google.colab import files

fpath = 'checkpoints/best_model_t5.pt'
if os.path.exists(fpath):
    files.download(fpath)
    print(f'✅ {fpath}  ({os.path.getsize(fpath)/1e6:.1f} MB)')
else:
    print(f'❌ 不存在: {fpath}')